In [1]:
import tensorflow_probability as tfp
import numpy as np
from tqdm.notebook import tqdm
import tensorflow as tf


tfd = tfp.distributions
tfb = tfp.bijectors
from bakeoff.TensorFlow_Prob.run_tfp import make_conditioned_lp

In [2]:
from modulars.utils import load_config, load_best_values, logistic_moments
from modulars.plot_rr import plot_a_few_trajectories_1d, plot_mean_band_rrs_1d
from modulars.tfp_rr_test import tfp_run_restart_1d
from modulars import binomial, beta_posterior
data_gen = binomial
transform_fn = logistic_moments

In [3]:
config_file = "../beta_config.json"
config = load_config(config_file)

theta_like = config['theta_like']
alpha_prior = config['alpha_prior']
beta_prior = config['beta_prior']
n_samples = config['n_samples']

data = data_gen(n_samples) if n_samples > 0 else 0

from modulars import print_model_info
print_model_info(
    "Beta", "Binomial", "Beta",
    [alpha_prior, beta_prior], [theta_like],
    data=data, posterior_func=beta_posterior,
    n_samples=n_samples)

Prior: Beta([0.5, 0.6])
Likelihood: Binomial([0.7])
Data: 7
Posterior: Beta({'alpha_post': 7.5, 'beta_post': 3.5999999999999996, 'mean_post': 0.6756756756756757, 'std_post': np.float64(0.13457556695458628)})


In [ ]:
def make_conditioned_lp_binomial_count(alpha_prior, beta_prior, total_count, y_obs, dtype=tf.float32):
    alpha = tf.convert_to_tensor(alpha_prior, dtype=dtype)
    beta  = tf.convert_to_tensor(beta_prior,  dtype=dtype)

    # Cast to float for TFP Binomial internal computations (avoids int32/float32 mismatch errors)
    total_count_f = tf.cast(total_count, dtype)
    y_obs_f       = tf.cast(y_obs, dtype)

    prior = tfd.Beta(concentration1=alpha, concentration0=beta)

    @tf.function(jit_compile=False)  # keep simple; VI can be jit-compiled separately
    def log_prob_fn(theta):
        theta = tf.convert_to_tensor(theta, dtype=dtype)  # theta in (0,1), vector shape [sample_size]
        lp = prior.log_prob(theta)
        lp += tfd.Binomial(total_count=total_count_f, probs=theta).log_prob(y_obs_f)
        return lp

    return log_prob_fn

conditioned_log_prob = make_conditioned_lp_binomial_count(
    alpha_prior=alpha_prior,
    beta_prior=beta_prior,
    total_count=n_samples,
    y_obs=data,
    dtype=tf.float32
)


alpha_prior = float(alpha_prior)
beta_prior  = float(beta_prior)
n_samples   = int(n_samples)
data        = int(data)

In [5]:
results = []
bij = tfb.Sigmoid()
for seed in tqdm(range(100)):
    result = tfp_run_restart_1d(
        seed, conditioned_log_prob, bij)
    results.append(result)

  0%|          | 0/100 [00:00<?, ?it/s]

2026-03-16 21:31:05.751658: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:108] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. fit_surrogate_posterior/sanitize_seed/seed
I0000 00:00:1773711066.206036 2118291 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [ ]:
%matplotlib inline
from modulars import apply_traj_transform, save_rr_tracking_csv

TRACKING_CSV = "processed_tracking/rr_tfp_beta_tracking.csv"

single_means, single_stds, multi_means, multi_stds = \
apply_traj_transform(results, transform_fn=transform_fn,
                     n_samples=100_000, seed=1, NOTEBOOK=True)
save_rr_tracking_csv(
    TRACKING_CSV,
    {"default": (single_means, single_stds, multi_means, multi_stds)},
)


In [ ]:
from modulars import load_rr_tracking_csv

TRACKING_CSV = "processed_tracking/rr_tfp_beta_tracking.csv"

# Re-run this cell to plot from the saved CSV without refitting or reprocessing.
single_means, single_stds, multi_means, multi_stds, x = load_rr_tracking_csv(
    TRACKING_CSV
)
N, T = single_means.shape

# If we have "best" from config, these should be on theta-scale (0,1)
best_mu, best_std = load_best_values(
    config=config, transform=transform_fn,
    n_samples=50_000, seed=1)


plot_a_few_trajectories_1d(
    [single_means, single_stds], [multi_means, multi_stds],
    best_mu, best_std, r'$\sigma^2$')
plot_mean_band_rrs_1d(
    single_means, single_stds, best_mu, best_std,
    x, r'$\sigma^2$', 1)
plot_mean_band_rrs_1d(
    multi_means, multi_stds, best_mu, best_std,
    x, r'$\sigma^2$', 100)
